# Accessing Data
- import.py
- relative ranking csv
- holiday function

## Import.py

In [4]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [5]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [6]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

## Relative ranking mean, StD, variance

In [8]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


# Plotting


## Mean
- mean relative rank  = y axis
- year = x axis
- each line represents a time block

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_mean_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/mean_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing mean relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot
    block_cols = {
        "00–04": "00_04_mean",
        "04–10": "04_10_mean",
        "10–15": "10_15_mean",
        "15–20": "15_20_mean",
        "20–24": "20_24_mean",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # --- NEW: full station name for title ---
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # --- NEW: title with full station name ---
            plt.title(f"{holiday} — Mean Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Mean Relative Rank", fontsize=14)
            plt.ylim(0, 1)
            plt.grid(alpha=0.3)

            # --- NEW: legend below the plot ---
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All plots saved.")


In [ ]:
plot_mean_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


2015 new year's day is empty from 1/01/15 - 4/01/15 due to NaN's in the raw data

## Variance

In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_variance_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/variance_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing variance of relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot (variance columns)
    block_cols = {
        "00–04": "00_04_var",
        "04–10": "04_10_var",
        "10–15": "10_15_var",
        "15–20": "15_20_var",
        "20–24": "20_24_var",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # Full station name for title
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # Title with full station name
            plt.title(f"{holiday} — Variance of Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Variance of Relative Rank", fontsize=14)
            plt.grid(alpha=0.3)

            # Legend below the plot
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All variance plots saved.")


In [13]:
plot_variance_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


All variance plots saved.


## StD

In [24]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_std_relative_rank_per_station(
    csv_path,
    out_base="/home/565/pv3484/aus_substation_electricity/figures/StD_relative_rank"
):
    """
    Reads the long-form daily relative-rank CSV and produces
    one plot per station per holiday, showing standard deviation of relative rank
    by year for the holiday ONLY (not the ±30-day window).
    """

    df = pd.read_csv(csv_path, parse_dates=["date"])

    # Filter to holiday-only rows
    df_h = df[df["is_holiday"] == True].copy()

    # Time blocks to plot (StD columns)
    block_cols = {
        "00–04": "00_04_std",
        "04–10": "04_10_std",
        "10–15": "10_15_std",
        "15–20": "15_20_std",
        "20–24": "20_24_std",
    }

    # Colours for consistency
    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # Loop through holidays
    for holiday in df_h["holiday"].unique():

        holiday_folder = os.path.join(out_base, holiday.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        df_hol = df_h[df_h["holiday"] == holiday]

        # Loop through stations
        for station in df_hol["station_code"].unique():

            df_s = df_hol[df_hol["station_code"] == station].sort_values("year")

            # Full station name for title
            full_name = df_s["station_name"].iloc[0]

            plt.figure(figsize=(10, 6))

            # Plot each block
            for label, col in block_cols.items():
                plt.plot(
                    df_s["year"],
                    df_s[col],
                    marker="o",
                    linewidth=2,
                    color=colours[label],
                    label=label
                )

            # Title with full station name
            plt.title(f"{holiday} — Standard Deviation of Relative Rank\n{full_name}", fontsize=16)

            plt.xlabel("Year", fontsize=14)
            plt.ylabel("Standard Deviation of Relative Rank", fontsize=14)
            plt.grid(alpha=0.3)

            # Legend below the plot
            plt.legend(
                title="Time Block (24hr)",
                fontsize=12,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=5,
                frameon=False
            )

            plt.tight_layout()

            out_path = os.path.join(holiday_folder, f"{station}.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close()

    print("All standard deviation plots saved.")


In [25]:
plot_std_relative_rank_per_station(
    csv_path="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv"
)


All standard deviation plots saved.


### 2x2 panel for powerpoint

In [18]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_std_panels_for_holiday(
    df,
    holiday,
    stations=["BLAKE", "GATES", "LIDCO", "CHATS"]
):
    """
    Produces a 2x2 panel of standard deviation plots for the given holiday
    for the four specified stations. Output is shown inline.
    """

    # Filter to holiday-only rows
    df_h = df[(df["is_holiday"] == True) & (df["holiday"] == holiday)].copy()

    # Standard deviation columns
    block_cols = {
        "00–04": "00_04_std",
        "04–10": "04_10_std",
        "10–15": "10_15_std",
        "15–20": "15_20_std",
        "20–24": "20_24_std",
    }

    colours = {
        "00–04": "red",
        "04–10": "orange",
        "10–15": "teal",
        "15–20": "blue",
        "20–24": "purple",
    }

    # 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, station in zip(axes, stations):

        df_s = df_h[df_h["station_code"] == station].sort_values("year")

        if df_s.empty:
            ax.set_title(f"{station} (no data)")
            ax.axis("off")
            continue

        full_name = df_s["station_name"].iloc[0]

        # Plot each time block
        for label, col in block_cols.items():
            ax.plot(
                df_s["year"],
                df_s[col],
                marker="o",
                linewidth=2,
                color=colours[label],
                label=label
            )

        ax.set_title(full_name, fontsize=14)
        ax.set_xlabel("Year")
        ax.set_ylabel("Standard Deviation of Relative Rank")
        ax.grid(alpha=0.3)

    # Shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Time Block (24hr)",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=5,
        frameon=False
    )

    fig.suptitle(f"{holiday} — Standard Deviation of Relative Rank", fontsize=18, y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.close()


In [19]:
# Load once
df = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/QLD/full_nsw_relative_rank.csv",
    parse_dates=["date"]
)

# Get all holidays in the dataset
holidays = df[df["is_holiday"] == True]["holiday"].unique()

# Loop and plot
for h in holidays:
    plot_std_panels_for_holiday(df, h)


# Troubleshooting

## Missing 2015 New Year's Day

In [ ]:
demand["2015-01-01":"2015-01-04"].isna().sum()
#checked for how many NaN values, with all substations outputting 5 = all substations were missing 5 values of relative ranked demand between 1/1/15 - 04/1/15
## was why when plotting, 2015 for NYD was skipped as original CSV code skipped any days with ANY NaN data points (which has now been corrected to skip any days with < 18 NaN values)

In [ ]:
demand["2015-01-01"]

In [ ]:
#demand.index.min()
demand.loc["2015-01"]

In [ ]:
demand[demand.index.year == 2015].head(10)


In [ ]:
demand.loc["2015-01-01"]


In [ ]:
demand["2015-01-01":"2015-01-04"].resample("D").count()


In [ ]:
demand["2015-01-01":"2015-01-04"].index.to_series().diff().value_counts()


In [ ]:
demand[demand.index.year == 2015].head(20)


In [ ]:
demand["2014-12-31":"2015-01-05"]

# Metadata tables

In [6]:
stations = ["BLAKEHURST", "GATESHEAD", "CHATSWOOD", "LIDCOMBE"]

info.loc[info["Name"].str.upper().isin(stations), 
         ["Name", "Residential", "Industrial"]]


,Name,Residential,Industrial
BLAKE,Blakehurst,0.850,0.021
CHATS,Chatswood,0.766,0.052
LIDCO,Lidcombe,0.743,0.080
GATES,Gateshead,0.479,0.057
